### **Installations and Imports**

In [1]:
import importlib.util
import subprocess
import sys
from importlib.metadata import version as pkg_version, PackageNotFoundError

def is_installed(import_name):
    return importlib.util.find_spec(import_name) is not None

# package_name_on_pip : import_name_in_python
required = {
    "unsloth": "unsloth",
    "unsloth_zoo": "unsloth_zoo",
    "trl": "trl",
    "bitsandbytes": "bitsandbytes",
    "sacrebleu": "sacrebleu",
    "evaluate": "evaluate",
    "packaging": "packaging",
}

missing = [
    pip_name
    for pip_name, import_name in required.items()
    if not is_installed(import_name)
]

print("Missing packages:", missing)

if missing:
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *missing,
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("All required extra packages are already installed.")

# Qwen3 needs recent transformers.
from packaging.version import parse as parse_version

def installed_version(package_name):
    try:
        return pkg_version(package_name)
    except PackageNotFoundError:
        return None

transformers_version = installed_version("transformers")
print("Current transformers:", transformers_version)

if transformers_version is None or parse_version(transformers_version) < parse_version("4.51.0"):
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "transformers>=4.51.0",
    ]
    print("Upgrading transformers for Qwen3 support:")
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("transformers version is OK for Qwen3.")

print("Minimal installation finished.")

Missing packages: ['unsloth', 'unsloth_zoo', 'trl', 'bitsandbytes', 'sacrebleu', 'evaluate']
Running: /usr/bin/python3 -m pip install -q --no-cache-dir unsloth unsloth_zoo trl bitsandbytes sacrebleu evaluate
Current transformers: 5.5.0
transformers version is OK for Qwen3.
Minimal installation finished.


In [2]:
# ============================================================
# Cell 1B — Verify environment
# ============================================================

import torch
import datasets
import transformers
import peft
import accelerate
import trl
import unsloth
import bitsandbytes
import sacrebleu

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available.")
    print("Do not start Qwen/Unsloth fine-tuning on CPU.")
    print("Go to Runtime → Change runtime type → GPU.")

print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("trl:", trl.__version__)
print("sacrebleu:", sacrebleu.__version__)

print("Environment check finished.")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
datasets: 4.3.0
transformers: 5.5.0
peft: 0.19.1
accelerate: 1.14.0
trl: 0.24.0
sacrebleu: 2.6.0
Environment check finished.


### **Paths and Configurations**

In [3]:
# ============================================================
# Cell 2 — Mount Google Drive and define paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/alexandria_qwen35_sft")
DATA_DIR    = PROJECT_DIR / "prepared_data"
RUNS_DIR    = PROJECT_DIR / "runs"
ADAPTER_DIR = PROJECT_DIR / "final_adapters"
PRED_DIR    = PROJECT_DIR / "predictions"

for p in [PROJECT_DIR, DATA_DIR, RUNS_DIR, ADAPTER_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("PRED_DIR:", PRED_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft
DATA_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data
RUNS_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/runs
ADAPTER_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters
PRED_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/predictions


In [4]:
# ============================================================
# Cell 3 — Experiment configuration
# Qwen3-4B, complete 2-shot prompt, ALL group, r=16, 10 epochs
# Save checkpoints every 100 steps, keep 50 checkpoints
# ============================================================

import torch
import random
import numpy as np

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen3-4B-Base"

# Qwen3-4B on Colab/T4 should use 4-bit.
LOAD_IN_4BIT = True

# ------------------------------------------------------------
# Country/config selection
# ------------------------------------------------------------

SELECTED_CONFIGS_MODE = "EG_ONLY"   # "EG_ONLY" or "ALL" or "MANUAL"
MANUAL_CONFIGS = ["EG"]

# ------------------------------------------------------------
# Translation setup
# ------------------------------------------------------------

MAX_CONTEXT_TURNS = 3
USE_PREVIOUS_ENGLISH_CONTEXT = True
USE_METADATA = True

# ------------------------------------------------------------
# Few-shot setup
# ------------------------------------------------------------

USE_FEW_SHOTS = True
N_FEW_SHOTS = 2

# We keep the selected examples COMPLETE.
# This value is only used to SELECT naturally short examples.
# It does NOT truncate the examples.
MAX_FEW_SHOT_EXAMPLE_CHARS = 450

# Reasonable default for Qwen3-4B LoRA on Colab/T4.
# If Cell 11 reports many examples longer than this, first reduce
# MAX_FEW_SHOT_EXAMPLE_CHARS before increasing MAX_SEQ_LENGTH.
MAX_SEQ_LENGTH = 2048

# ------------------------------------------------------------
# LoRA mode
# ------------------------------------------------------------
# "attn" = q_proj, k_proj, v_proj, o_proj
# "mlp"  = FNN / feed-forward group: gate_proj, up_proj, down_proj
# "all"  = attention + FNN / MLP

LORA_MODE = "all"

# ------------------------------------------------------------
# LoRA rank settings
# ------------------------------------------------------------

LORA_R = 16
LORA_ALPHA = 32

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

NUM_EPOCHS = 10

PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03

SAVE_STEPS = 100
EVAL_STEPS = 100
LOGGING_STEPS = 10

PACKING = False

SAVE_TOTAL_LIMIT = 50

# ------------------------------------------------------------
# Experiment naming
# ------------------------------------------------------------

if LORA_MODE == "attn":
    LORA_GROUP_NAME = "attn_group"
elif LORA_MODE == "mlp":
    LORA_GROUP_NAME = "fnn_group"
elif LORA_MODE == "all":
    LORA_GROUP_NAME = "all_group"
else:
    raise ValueError("LORA_MODE must be 'attn', 'mlp', or 'all'.")

# No v2.
# This is the first Qwen3-4B complete-2shot experiment.
EXPERIMENT_NAME = (
    f"qwen3_4b_alexandria_{SELECTED_CONFIGS_MODE.lower()}_"
    f"context{MAX_CONTEXT_TURNS}_complete2shot_{LORA_GROUP_NAME}_r{LORA_R}_10epochs"
)

OUTPUT_DIR = RUNS_DIR / EXPERIMENT_NAME
FINAL_ADAPTER_PATH = ADAPTER_DIR / EXPERIMENT_NAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ADAPTER_PATH.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Load in 4-bit:", LOAD_IN_4BIT)
print("Experiment:", EXPERIMENT_NAME)
print("LoRA mode:", LORA_MODE)
print("LoRA r:", LORA_R)
print("LoRA alpha:", LORA_ALPHA)
print("Few-shot enabled:", USE_FEW_SHOTS)
print("Few-shot examples:", N_FEW_SHOTS)
print("Max few-shot example chars for selection only:", MAX_FEW_SHOT_EXAMPLE_CHARS)
print("Max seq length:", MAX_SEQ_LENGTH)
print("Save steps:", SAVE_STEPS)
print("Eval steps:", EVAL_STEPS)
print("Save total limit:", SAVE_TOTAL_LIMIT)
print("Output dir:", OUTPUT_DIR)
print("Final adapter:", FINAL_ADAPTER_PATH)

Model: Qwen/Qwen3-4B-Base
Load in 4-bit: True
Experiment: qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs
LoRA mode: all
LoRA r: 16
LoRA alpha: 32
Few-shot enabled: True
Few-shot examples: 2
Max few-shot example chars for selection only: 450
Max seq length: 2048
Save steps: 100
Eval steps: 100
Save total limit: 50
Output dir: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs
Final adapter: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs


### **Dataset Preparation**

In [5]:
# ============================================================
# Cell 4 — List Alexandria configs and load selected configs
# ============================================================

from datasets import load_dataset, get_dataset_config_names
import pandas as pd

DATASET_NAME = "UBC-NLP/alexandria"

available_configs = get_dataset_config_names(DATASET_NAME)
print("Available Alexandria configs:")
print(available_configs)

if SELECTED_CONFIGS_MODE == "EG_ONLY":
    selected_configs = ["EG"] if "EG" in available_configs else [available_configs[0]]

elif SELECTED_CONFIGS_MODE == "ALL":
    selected_configs = available_configs

elif SELECTED_CONFIGS_MODE == "MANUAL":
    selected_configs = MANUAL_CONFIGS
    missing = [c for c in selected_configs if c not in available_configs]
    if missing:
        raise ValueError(f"These configs are not available: {missing}")

else:
    raise ValueError("SELECTED_CONFIGS_MODE must be EG_ONLY, ALL, or MANUAL.")

print("\nSelected configs:")
print(selected_configs)

loaded = {}

for cfg in selected_configs:
    print(f"\nLoading config: {cfg}")
    ds_train = load_dataset(DATASET_NAME, name=cfg, split="train")
    ds_test  = load_dataset(DATASET_NAME, name=cfg, split="test")

    loaded[cfg] = {
        "train": ds_train,
        "test": ds_test,
    }

    print("Train:", ds_train)
    print("Test:", ds_test)
    print("Example keys:", ds_train[0].keys())

README.md:   0%|          | 0.00/25.2k [00:00<?, ?B/s]

Available Alexandria configs:
['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']

Selected configs:
['EG']

Loading config: EG


EG/train-00000-of-00001.parquet:   0%|          | 0.00/496k [00:00<?, ?B/s]

EG/test-00000-of-00001.parquet:   0%|          | 0.00/196k [00:00<?, ?B/s]

EG/dev-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/982 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/366 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/352 [00:00<?, ? examples/s]

Train: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 982
})
Test: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 366
})
Example keys: dict_keys(['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'])


In [6]:
# ============================================================
# Cell 5 — Inspect one raw example
# ============================================================

sample_cfg = selected_configs[0]
sample_row = loaded[sample_cfg]["train"][0]

print("Config:", sample_cfg)
print("Keys:", sample_row.keys())

print("\nEnglish conversation:")
print(sample_row["english_conversation"])

print("\nDialectal conversation:")
print(sample_row["dialectal_conversation"])

print("\nFull row:")
sample_row

Config: EG
Keys: dict_keys(['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'])

English conversation:
[{'direction': 'male -> female', 'speaker': 'Wholesale Buyer', 'text': "Good morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?", 'turn_order': 1}, {'direction': 'female -> male', 'speaker': 'Wholesale Seller', 'text': "Good morning to you. You heard correctly. My artichokes are the best you'll find. They are top-grade, perfect for export. Let me show you a sample.", 'turn_order': 2}, {'direction': 'male -> female', 'speaker': 'Wholesale Buyer', 'text': 'Excellent. Yes, please show me. I need them to be a specific size and completely free of blemishes.', 'turn_order': 3}, {'direction': 'female -> male', 'speaker': 'Wholesale Seller', 'text': "Don't you worry. You will be very satisfied. My reputatio

{'conv_id': 'B7-1-0-120',
 'country': 'EG',
 'domain': 'Agriculture and farming',
 'dialect': 'Egyptian Arabic (Cairene) Dialect',
 'participants': 'Wholesale Buyer, Wholesale Seller',
 'english_conversation': [{'direction': 'male -> female',
   'speaker': 'Wholesale Buyer',
   'text': "Good morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?",
   'turn_order': 1},
  {'direction': 'female -> male',
   'speaker': 'Wholesale Seller',
   'text': "Good morning to you. You heard correctly. My artichokes are the best you'll find. They are top-grade, perfect for export. Let me show you a sample.",
   'turn_order': 2},
  {'direction': 'male -> female',
   'speaker': 'Wholesale Buyer',
   'text': 'Excellent. Yes, please show me. I need them to be a specific size and completely free of blemishes.',
   'turn_order': 3},
  {'direction': 'female -> male',
   'speaker': 'Wholesale Seller',
   'text': "D

#### Helper functions for robust extraction

In [7]:
# ============================================================
# Cell 6 — Helper functions for robust extraction
# ============================================================

def safe_get(row, keys, default=""):
    for k in keys:
        if isinstance(row, dict) and k in row and row[k] is not None:
            return row[k]
    return default

def turn_text(turn):
    if isinstance(turn, dict):
        for k in ["text", "sentence", "utterance", "content", "value"]:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
        return str(turn).strip()
    return str(turn).strip()

def turn_field(turn, keys, default=""):
    if isinstance(turn, dict):
        for k in keys:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
    return default

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return list(x)

def truncate_text(text, max_chars=1200):
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ..."

#### Flatten Alexandria conversations into SFT examples and Save

In [8]:
# ============================================================
# Cell 7 — Flatten Alexandria conversations
# ============================================================

def flatten_alexandria_split(ds, cfg_name, split_name, max_context_turns=3):
    records = []

    for conv_idx, row in enumerate(ds):
        english_conv = normalize_list(row["english_conversation"])
        dialect_conv = normalize_list(row["dialectal_conversation"])

        n = min(len(english_conv), len(dialect_conv))

        country = safe_get(row, ["country", "country_code"], cfg_name)
        dialect = safe_get(row, ["dialect", "dialect_label", "subdialect", "city", "variety"], "")
        domain = safe_get(row, ["domain", "topic"], "")
        persona = safe_get(row, ["persona", "roles", "speaker_roles"], "")
        conv_id = safe_get(row, ["conversation_id", "id", "dialogue_id"], f"{cfg_name}_{split_name}_{conv_idx}")

        for i in range(n):
            en_turn = english_conv[i]
            ar_turn = dialect_conv[i]

            source_text = turn_text(en_turn)
            target_text = turn_text(ar_turn)

            if not source_text or not target_text:
                continue

            prev_start = max(0, i - max_context_turns)
            prev_en_turns = english_conv[prev_start:i]

            previous_context = []
            for j, t in enumerate(prev_en_turns, start=prev_start):
                previous_context.append({
                    "turn_id": j,
                    "speaker": turn_field(t, ["speaker", "role", "speaker_role"], ""),
                    "direction": turn_field(t, ["direction", "gender_direction", "speaker_addressee_gender"], ""),
                    "text": turn_text(t),
                })

            records.append({
                "source_id": f"{cfg_name}_{split_name}_{conv_id}_{i}",
                "config": cfg_name,
                "split": split_name,
                "conversation_id": conv_id,
                "turn_id": i,

                "country": country,
                "dialect": dialect,
                "domain": domain,
                "persona": persona,

                "speaker": turn_field(en_turn, ["speaker", "role", "speaker_role"], ""),
                "gender_direction": turn_field(en_turn, ["direction", "gender_direction", "speaker_addressee_gender"], ""),

                "previous_english_turns": previous_context,
                "source_text": source_text,
                "target_arabic": target_text,
            })

    return records

train_records = []
eval_records = []

for cfg in selected_configs:
    train_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["train"],
            cfg_name=cfg,
            split_name="train",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

    eval_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["test"],
            cfg_name=cfg,
            split_name="test",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

train_df = pd.DataFrame(train_records)
eval_df = pd.DataFrame(eval_records)

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)

print("\nTrain configs:")
print(train_df["config"].value_counts())

print("\nEval configs:")
print(eval_df["config"].value_counts())

display(train_df.head())

Train shape: (3108, 14)
Eval shape: (1118, 14)

Train configs:
config
EG    3108
Name: count, dtype: int64

Eval configs:
config
EG    1118
Name: count, dtype: int64


,source_id,config,split,conversation_id,turn_id,country,dialect,domain,persona,speaker,gender_direction,previous_english_turns,source_text,target_arabic
0,EG_train_EG_train_0_0,EG,train,EG_train_0,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,[],Good morning. I'm looking to source 10 tons of...,صباح الخير، عايز عشرة طن من الخرشوف الكويس للت...
1,EG_train_EG_train_0_1,EG,train,EG_train_0,1,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Good morning to you. You heard correctly. My a...,صباح النور،سمعك مظبوط،الخرشوف بتاعي من أحسن ال...
2,EG_train_EG_train_0_2,EG,train,EG_train_0,2,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...","Excellent. Yes, please show me. I need them to...",ممتاز، لو سمحتي وريني، عايزه بمقاس واحد ومافيه...
3,EG_train_EG_train_0_3,EG,train,EG_train_0,3,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Don't you worry. You will be very satisfied. M...,متخافش، هتنبسط جدا، سمعتي جاية من الحاجة الكويسة.
4,EG_train_EG_train_1_0,EG,train,EG_train_1,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Farmer,female -> female,[],I usually use the regular granular fertilizer....,أنا عادة بستخدم السماد العادي الحبيبات. ايه فا...


In [9]:
# ============================================================
# Cell 8 — Save prepared flattened data
# ============================================================

train_jsonl = DATA_DIR / f"alexandria_train_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"
eval_jsonl  = DATA_DIR / f"alexandria_eval_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"

train_df.to_json(train_jsonl, orient="records", lines=True, force_ascii=False)
eval_df.to_json(eval_jsonl, orient="records", lines=True, force_ascii=False)

print("Saved train:", train_jsonl)
print("Saved eval:", eval_jsonl)

Saved train: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_train_eg_only_context3.jsonl
Saved eval: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_eval_eg_only_context3.jsonl


### **Build prompt and chat messages**

In [10]:
# ============================================================
# Cell 9 — Build prompt and chat messages
# Complete 2-shot examples from training data only
# ============================================================

from datasets import Dataset
import hashlib
import pandas as pd

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Use the provided training examples only as style and dialect guidance. "
    "Return only the translation, without explanation."
)

def build_context(previous_turns):
    if not USE_PREVIOUS_ENGLISH_CONTEXT or not previous_turns:
        return "No previous context."

    lines = []
    for i, t in enumerate(previous_turns, start=1):
        speaker = t.get("speaker", "")
        text = t.get("text", "")

        if speaker:
            lines.append(f"{i}. {speaker}: {text}")
        else:
            lines.append(f"{i}. {text}")

    return "\n".join(lines)

def build_metadata_block(row):
    if not USE_METADATA:
        return "No metadata."

    fields = [
        ("Country/config", row.get("config", "")),
        ("Target dialect", row.get("dialect", "")),
        ("Domain", row.get("domain", "")),
        ("Persona/Roles", row.get("persona", "")),
        ("Current speaker", row.get("speaker", "")),
        ("Speaker-to-addressee gender direction", row.get("gender_direction", "")),
    ]

    lines = []
    for k, v in fields:
        v = str(v).strip()
        if v:
            lines.append(f"{k}: {v}")

    return "\n".join(lines) if lines else "No metadata."

def deterministic_seed_from_id(source_id, base_seed=SEED):
    raw = f"{source_id}_{base_seed}".encode("utf-8")
    return int(hashlib.md5(raw).hexdigest()[:8], 16)

def select_two_shots_from_train(row, train_pool, n=N_FEW_SHOTS):
    """
    Select n COMPLETE examples from training data only.

    The examples are NOT truncated.

    To keep MAX_SEQ_LENGTH reasonable, we prefer naturally short examples:
        source_text length + target_arabic length <= MAX_FEW_SHOT_EXAMPLE_CHARS

    Priority:
    1. same config + same domain + short
    2. same config + short
    3. any short
    4. same config + same domain
    5. same config
    6. any training example

    For train rows, exclude the same source_id to avoid using itself as a shot.
    """
    if not USE_FEW_SHOTS or n <= 0:
        return []

    row_source_id = str(row.get("source_id", ""))
    row_config = str(row.get("config", ""))
    row_domain = str(row.get("domain", ""))

    pool = train_pool.copy()
    pool["source_id"] = pool["source_id"].astype(str)

    # Avoid using the same train row as its own few-shot example.
    pool = pool[pool["source_id"] != row_source_id].copy()

    if len(pool) == 0:
        return []

    pool["fewshot_total_chars"] = (
        pool["source_text"].astype(str).str.len()
        + pool["target_arabic"].astype(str).str.len()
    )

    short_pool = pool[pool["fewshot_total_chars"] <= MAX_FEW_SHOT_EXAMPLE_CHARS].copy()

    same_config_domain_short = short_pool[
        (short_pool["config"].astype(str) == row_config)
        & (short_pool["domain"].astype(str) == row_domain)
    ]

    same_config_short = short_pool[
        short_pool["config"].astype(str) == row_config
    ]

    same_config_domain = pool[
        (pool["config"].astype(str) == row_config)
        & (pool["domain"].astype(str) == row_domain)
    ]

    same_config = pool[
        pool["config"].astype(str) == row_config
    ]

    candidate_pools = [
        same_config_domain_short,
        same_config_short,
        short_pool,
        same_config_domain,
        same_config,
        pool,
    ]

    candidates = None
    for candidate_pool in candidate_pools:
        if len(candidate_pool) >= n:
            candidates = candidate_pool
            break

    if candidates is None:
        candidates = pool

    sample_n = min(n, len(candidates))
    seed = deterministic_seed_from_id(row_source_id)

    shots = candidates.sample(n=sample_n, random_state=seed)

    keep_cols = [
        "source_id",
        "config",
        "dialect",
        "domain",
        "source_text",
        "target_arabic",
    ]

    return shots[keep_cols].to_dict("records")

def build_few_shot_block(few_shot_examples):
    if not USE_FEW_SHOTS or not few_shot_examples:
        return "No examples available."

    blocks = []

    for i, ex in enumerate(few_shot_examples, start=1):
        ex_config = str(ex.get("config", "")).strip()
        ex_dialect = str(ex.get("dialect", "")).strip()
        ex_domain = str(ex.get("domain", "")).strip()

        meta_parts = []
        if ex_config:
            meta_parts.append(f"config={ex_config}")
        if ex_dialect:
            meta_parts.append(f"dialect={ex_dialect}")
        if ex_domain:
            meta_parts.append(f"domain={ex_domain}")

        meta_line = ", ".join(meta_parts) if meta_parts else "no metadata"

        # COMPLETE examples.
        # No truncation here.
        ex_source = str(ex.get("source_text", "")).strip()
        ex_target = str(ex.get("target_arabic", "")).strip()

        blocks.append(
            f"""Example {i} ({meta_line})
English:
{ex_source}

Arabic:
{ex_target}"""
        )

    return "\n\n".join(blocks)

def make_user_prompt(row):
    context = build_context(row["previous_english_turns"])
    metadata = build_metadata_block(row)
    few_shots = build_few_shot_block(row.get("few_shot_examples", []))

    return f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
{few_shots}

Metadata:
{metadata}

Previous English dialogue context:
{context}

Current English turn:
{row["source_text"]}

Rules:
- Preserve the meaning exactly.
- Use the target local dialect, not Modern Standard Arabic unless it is natural in context.
- Follow the dialect/style pattern shown in the few-shot examples when relevant.
- Do not copy the few-shot examples.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""

def row_to_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": row["target_arabic"]},
    ]

# ------------------------------------------------------------
# Build few-shot pool from training data only
# ------------------------------------------------------------

few_shot_pool_cols = [
    "source_id",
    "config",
    "dialect",
    "domain",
    "source_text",
    "target_arabic",
]

missing_few_shot_cols = [
    c for c in few_shot_pool_cols
    if c not in train_df.columns
]

if missing_few_shot_cols:
    raise ValueError(f"Missing columns in train_df for few-shot selection: {missing_few_shot_cols}")

train_few_shot_pool = train_df[few_shot_pool_cols].copy()

train_df["few_shot_examples"] = train_df.apply(
    lambda row: select_two_shots_from_train(row, train_few_shot_pool, n=N_FEW_SHOTS),
    axis=1,
)

eval_df["few_shot_examples"] = eval_df.apply(
    lambda row: select_two_shots_from_train(row, train_few_shot_pool, n=N_FEW_SHOTS),
    axis=1,
)

train_df["messages"] = train_df.apply(row_to_messages, axis=1)
eval_df["messages"]  = eval_df.apply(row_to_messages, axis=1)

train_dataset = Dataset.from_pandas(
    train_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

eval_dataset = Dataset.from_pandas(
    eval_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

print(train_dataset)
print(eval_dataset)

print("\nExample few-shot source IDs for first train row:")
print([x["source_id"] for x in train_df.iloc[0]["few_shot_examples"]])

print("\nFew-shot example lengths for first train row:")
for i, ex in enumerate(train_df.iloc[0]["few_shot_examples"], start=1):
    total_chars = len(str(ex["source_text"])) + len(str(ex["target_arabic"]))
    print(f"Example {i}: source_id={ex['source_id']}, total_chars={total_chars}")

print("\nExample messages:")
train_dataset[0]["messages"]

Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 3108
})
Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 1118
})

Example few-shot source IDs for first train row:
['EG_train_EG_train_46_2', 'EG_train_EG_train_67_0']

Few-shot example lengths for first train row:
Example 1: source_id=EG_train_EG_train_46_2, total_chars=243
Example 2: source_id=EG_train_EG_train_67_0, total_chars=75

Example messages:


[{'role': 'system',
  'content': 'You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Use the provided training examples only as style and dialect guidance. Return only the translation, without explanation.'},
 {'role': 'user',
  'content': "Task:\nTranslate the current English dialogue turn into the target dialectal Arabic variety.\n\nFew-shot training examples:\nExample 1 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)\nEnglish:\nBy monitoring, we'll know exactly when the pest levels are high enough to justify spraying. This saves you money and protects the environment.\n\nArabic:\nلما نراقب، هنعرف بالضبط امتى مستويات الحشرات عالية كفاية عشان نبرر الرش. ده بيوفر فلوس وبيحمي البيئة.\n\nExample 2 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)\nEnglish:\nSo, what will this cost me for the whole job?\n\nArabic:\nطيب قد ايه الشغل كله هيكل

### **Load Qwen3-4B with Unsloth**

In [11]:
# ============================================================
# UPDATED Cell 10 — Load Qwen3-4B with Unsloth
# Prefer FastLanguageModel for text-only SFT
# ============================================================

import torch

try:
    from unsloth import FastLanguageModel
    UnslothModel = FastLanguageModel
    print("Using unsloth.FastLanguageModel")
except Exception:
    from unsloth import FastModel
    UnslothModel = FastModel
    print("Using unsloth.FastModel fallback")

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

model, tokenizer = UnslothModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = dtype,
    load_in_4bit = LOAD_IN_4BIT,
)

# If Unsloth returns a processor-like object, extract the real tokenizer if available.
if hasattr(tokenizer, "tokenizer"):
    print("Tokenizer object has internal tokenizer. Using tokenizer.tokenizer.")
    tokenizer = tokenizer.tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded:", MODEL_NAME)
print("dtype:", dtype)
print("load_in_4bit:", LOAD_IN_4BIT)
print("Tokenizer type:", type(tokenizer))
print("pad token:", tokenizer.pad_token)
print("eos token:", tokenizer.eos_token)

Using unsloth.FastLanguageModel
==((====))==  Unsloth 2026.6.9: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.32G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen3-4b-base-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loaded: Qwen/Qwen3-4B-Base
dtype: torch.float16
load_in_4bit: True
Tokenizer type: <class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
pad token: <|PAD_TOKEN|>
eos token: <|endoftext|>


### **Apply Chat Template**

In [12]:
# ============================================================
# Cell 11 — Manual SFT template + token length check
# Do NOT use tokenizer.apply_chat_template
# ============================================================

SYSTEM_MARKER = "### System:"
INSTRUCTION_MARKER = "### Instruction:"
RESPONSE_MARKER = "### Arabic translation:"

def get_message_content(messages, role):
    for m in messages:
        if m.get("role") == role:
            return m.get("content", "")
    return ""

def format_sft_text(system_text, user_text, assistant_text=None, add_eos=True):
    """
    Manual decoder-only SFT format.

    During training:
        prompt + assistant answer + EOS

    During inference:
        prompt only, ending at RESPONSE_MARKER
    """

    text = (
        f"{SYSTEM_MARKER}\n"
        f"{system_text.strip()}\n\n"
        f"{INSTRUCTION_MARKER}\n"
        f"{user_text.strip()}\n\n"
        f"{RESPONSE_MARKER}\n"
    )

    if assistant_text is not None:
        text += assistant_text.strip()

        if add_eos and tokenizer.eos_token is not None:
            text += tokenizer.eos_token

    return text

def apply_manual_sft_template(example):
    messages = example["messages"]

    system_text = get_message_content(messages, "system")
    user_text = get_message_content(messages, "user")
    assistant_text = get_message_content(messages, "assistant")

    text = format_sft_text(
        system_text=system_text,
        user_text=user_text,
        assistant_text=assistant_text,
        add_eos=True,
    )

    return {"text": text}

remove_cols_train = [c for c in train_dataset.column_names if c == "messages"]
remove_cols_eval  = [c for c in eval_dataset.column_names if c == "messages"]

train_dataset_text = train_dataset.map(
    apply_manual_sft_template,
    remove_columns=remove_cols_train,
)

eval_dataset_text = eval_dataset.map(
    apply_manual_sft_template,
    remove_columns=remove_cols_eval,
)

print(train_dataset_text)
print(eval_dataset_text)

print("\nFormatted example:")
print(train_dataset_text[0]["text"])

# ------------------------------------------------------------
# Token length diagnostics for complete 2-shot setup
# ------------------------------------------------------------

def count_tokens(example):
    ids = tokenizer(
        example["text"],
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    return {"n_tokens": len(ids)}

train_dataset_text = train_dataset_text.map(
    count_tokens,
    desc="Counting train tokens",
)

eval_dataset_text = eval_dataset_text.map(
    count_tokens,
    desc="Counting eval tokens",
)

train_lengths = train_dataset_text["n_tokens"]
eval_lengths = eval_dataset_text["n_tokens"]

def summarize_lengths(lengths, name):
    s = pd.Series(lengths)
    print(f"\n{name} token length summary:")
    print("count:", len(s))
    print("min:", int(s.min()))
    print("median:", int(s.median()))
    print("p90:", int(s.quantile(0.90)))
    print("p95:", int(s.quantile(0.95)))
    print("p99:", int(s.quantile(0.99)))
    print("max:", int(s.max()))

summarize_lengths(train_lengths, "Train")
summarize_lengths(eval_lengths, "Eval")

max_train_len = max(train_lengths)
max_eval_len = max(eval_lengths)
max_seen_len = max(max_train_len, max_eval_len)

too_long_train = sum(x > MAX_SEQ_LENGTH for x in train_lengths)
too_long_eval = sum(x > MAX_SEQ_LENGTH for x in eval_lengths)

print("\nMAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("Maximum complete 2-shot token length:", max_seen_len)

print("\nTrain examples longer than MAX_SEQ_LENGTH:", too_long_train)
print("Eval examples longer than MAX_SEQ_LENGTH:", too_long_eval)

if too_long_train > 0 or too_long_eval > 0:
    print("\nWARNING:")
    print("Some examples are longer than MAX_SEQ_LENGTH.")
    print("Recommended first fix:")
    print("Decrease MAX_FEW_SHOT_EXAMPLE_CHARS in Cell 3, for example:")
    print("MAX_FEW_SHOT_EXAMPLE_CHARS = 350")
    print("\nOnly increase MAX_SEQ_LENGTH if many useful examples are still too long.")
else:
    print("\nOK: MAX_SEQ_LENGTH is enough for the selected complete 2-shot setup.")

# ------------------------------------------------------------
# Optional: show longest examples for debugging
# ------------------------------------------------------------

train_len_df = pd.DataFrame({
    "idx": list(range(len(train_lengths))),
    "n_tokens": train_lengths,
}).sort_values("n_tokens", ascending=False)

eval_len_df = pd.DataFrame({
    "idx": list(range(len(eval_lengths))),
    "n_tokens": eval_lengths,
}).sort_values("n_tokens", ascending=False)

print("\nTop 5 longest train examples:")
print(train_len_df.head(5))

print("\nTop 5 longest eval examples:")
print(eval_len_df.head(5))

Map:   0%|          | 0/3108 [00:00<?, ? examples/s]

Map:   0%|          | 0/1118 [00:00<?, ? examples/s]

Dataset({
    features: ['source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic', 'text'],
    num_rows: 3108
})
Dataset({
    features: ['source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic', 'text'],
    num_rows: 1118
})

Formatted example:
### System:
You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Use the provided training examples only as style and dialect guidance. Return only the translation, without explanation.

### Instruction:
Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
Example 1 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)
English:
By monitoring, we'll know exactly when the pest levels are high enough to justify spraying. This saves you money and protects the environment.

Arabic:
لما نراقب، هنعرف بالضبط امتى مستويات الحشرات عالية كفاية عشا

Counting train tokens:   0%|          | 0/3108 [00:00<?, ? examples/s]

Counting eval tokens:   0%|          | 0/1118 [00:00<?, ? examples/s]


Train token length summary:
count: 3108
min: 354
median: 470
p90: 541
p95: 561
p99: 594
max: 692

Eval token length summary:
count: 1118
min: 348
median: 469
p90: 542
p95: 558
p99: 604
max: 665

MAX_SEQ_LENGTH: 2048
Maximum complete 2-shot token length: 692

Train examples longer than MAX_SEQ_LENGTH: 0
Eval examples longer than MAX_SEQ_LENGTH: 0

OK: MAX_SEQ_LENGTH is enough for the selected complete 2-shot setup.

Top 5 longest train examples:
       idx  n_tokens
1520  1520       692
3031  3031       626
814    814       625
2350  2350       624
2944  2944       623

Top 5 longest eval examples:
     idx  n_tokens
568  568       665
556  556       645
352  352       644
709  709       640
649  649       639


#### **Configure LoRA target modules**

In [13]:
# ============================================================
# Cell 12 — Configure LoRA
# Safe rerun: ALL group, r=16
# ============================================================

if LORA_MODE == "attn":
    TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

elif LORA_MODE == "mlp":
    TARGET_MODULES = ["gate_proj", "up_proj", "down_proj"]

elif LORA_MODE == "all":
    TARGET_MODULES = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]

else:
    raise ValueError("LORA_MODE must be 'attn', 'mlp', or 'all'.")

print("Target modules:", TARGET_MODULES)
print("LoRA r:", LORA_R)
print("LoRA alpha:", LORA_ALPHA)

model = UnslothModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    max_seq_length = MAX_SEQ_LENGTH,
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
LoRA r: 16
LoRA alpha: 32


Unsloth 2026.6.9 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


#### Check for existing checkpoints

In [14]:
# ============================================================
# Cell 13 — Check for existing checkpoints
# ============================================================

from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = None

if OUTPUT_DIR.exists():
    last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))

if last_checkpoint:
    print("Found checkpoint:")
    print(last_checkpoint)
else:
    print("No checkpoint found. Training will start from scratch.")

Found checkpoint:
/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-2000


### **Build SFTTrainer**

In [15]:
# ============================================================
# Cell 14 — Build SFTTrainer
# ============================================================

from trl import SFTTrainer, SFTConfig

sft_args = SFTConfig(
    output_dir = str(OUTPUT_DIR),

    num_train_epochs = NUM_EPOCHS,
    per_device_train_batch_size = PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size = 1,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,

    learning_rate = LEARNING_RATE,
    warmup_ratio = WARMUP_RATIO,
    lr_scheduler_type = "cosine",

    optim = "adamw_8bit",
    weight_decay = 0.01,

    logging_steps = LOGGING_STEPS,

    eval_strategy = "steps",
    eval_steps = EVAL_STEPS,

    save_strategy = "steps",
    save_steps = SAVE_STEPS,
    save_total_limit = SAVE_TOTAL_LIMIT,

    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),

    seed = SEED,
    dataset_num_proc = 2,
    report_to = "none",

    packing = PACKING,
    dataset_text_field = "text",
    max_length = MAX_SEQ_LENGTH,
)

try:
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset_text,
        eval_dataset = eval_dataset_text,
        args = sft_args,
    )
except TypeError:
    trainer = SFTTrainer(
        model = model,
        processing_class = tokenizer,
        train_dataset = train_dataset_text,
        eval_dataset = eval_dataset_text,
        args = sft_args,
    )

print("Trainer ready.")
print("Output dir:", OUTPUT_DIR)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3108 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1118 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer ready.
Output dir: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs


#### Response-only training with manual markers

In [16]:
# ============================================================
# Cell 15 — Train on assistant response only
# Robust response-marker version with marker verification
# ============================================================

def safe_len(x):
    try:
        return len(x)
    except Exception:
        return None

print("Before response-only masking:")
print("trainer.train_dataset length:", safe_len(trainer.train_dataset))
print("trainer.eval_dataset length:", safe_len(trainer.eval_dataset))

if "trainer" not in globals():
    raise NameError("trainer is not defined. Run Cell 14 before Cell 15.")

if "INSTRUCTION_MARKER" not in globals():
    raise NameError("INSTRUCTION_MARKER is not defined. Run Cell 11 before Cell 15.")

if "RESPONSE_MARKER" not in globals():
    raise NameError("RESPONSE_MARKER is not defined. Run Cell 11 before Cell 15.")

if safe_len(trainer.train_dataset) == 0:
    raise ValueError(
        "trainer.train_dataset is already empty before response-only masking. "
        "This means SFTTrainer preprocessing removed all samples. "
        "Check Cell 11 token-length output and MAX_SEQ_LENGTH."
    )

# ------------------------------------------------------------
# Verify that the response marker exists in formatted text
# ------------------------------------------------------------

instruction_marker_for_masking = f"{INSTRUCTION_MARKER}\n"
response_marker_for_masking = f"{RESPONSE_MARKER}\n"

print("\nMarkers used for masking:")
print("Instruction marker:", repr(instruction_marker_for_masking))
print("Response marker:", repr(response_marker_for_masking))

sample_text = None

if "train_dataset_text" in globals() and len(train_dataset_text) > 0:
    sample_text = train_dataset_text[0]["text"]
else:
    try:
        sample_text = trainer.train_dataset[0]["text"]
    except Exception:
        sample_text = None

if sample_text is None:
    raise ValueError(
        "Could not inspect a sample formatted text. "
        "Make sure Cell 11 and Cell 14 ran correctly."
    )

print("\nChecking first formatted training sample...")

if instruction_marker_for_masking not in sample_text:
    raise ValueError(
        "Instruction marker was not found in the formatted text. "
        "Check INSTRUCTION_MARKER and Cell 11 formatting."
    )

if response_marker_for_masking not in sample_text:
    raise ValueError(
        "Response marker was not found in the formatted text. "
        "Check RESPONSE_MARKER and Cell 11 formatting."
    )

print("Instruction marker found:", instruction_marker_for_masking in sample_text)
print("Response marker found:", response_marker_for_masking in sample_text)

# ------------------------------------------------------------
# Apply response-only masking
# ------------------------------------------------------------

try:
    from unsloth.chat_templates import train_on_responses_only

    trainer = train_on_responses_only(
        trainer,
        instruction_part=instruction_marker_for_masking,
        response_part=response_marker_for_masking,
    )

    print("\nEnabled response-only training.")

    print("\nAfter response-only masking:")
    print("trainer.train_dataset length:", safe_len(trainer.train_dataset))
    print("trainer.eval_dataset length:", safe_len(trainer.eval_dataset))

    if safe_len(trainer.train_dataset) == 0:
        raise ValueError(
            "Response-only masking produced an empty train dataset. "
            "This means the response marker was not found after tokenization, "
            "or all valid labels were removed."
        )

except Exception as e:
    print("\nCould not enable response-only training safely.")
    print("Reason:", repr(e))
    print("\nStopping here instead of silently doing wrong full-text SFT.")
    raise

Before response-only masking:
trainer.train_dataset length: 3108
trainer.eval_dataset length: 1118

Markers used for masking:
Instruction marker: '### Instruction:\n'
Response marker: '### Arabic translation:\n'

Checking first formatted training sample...
Instruction marker found: True
Response marker found: True


Map (num_proc=6):   0%|          | 0/3108 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/3108 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/1118 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/1118 [00:00<?, ? examples/s]


Enabled response-only training.

After response-only masking:
trainer.train_dataset length: 3108
trainer.eval_dataset length: 1118


### **Training**

In [17]:
# ============================================================
# Cell 16 — Train or resume
# ============================================================

if last_checkpoint:
    print("Resuming from:", last_checkpoint)
    trainer_stats = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting from scratch.")
    trainer_stats = trainer.train()

print("Training finished.")
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Resuming from: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-2000


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,108 | Num Epochs = 10 | Total steps = 3,890
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


KeyboardInterrupt: 

### **Save final LoRA adapter**

In [18]:
# ============================================================
# Cell 17 — Find, load, and save BEST checkpoint
# ============================================================

from pathlib import Path
import json
import torch

def find_best_checkpoint_from_logs(output_dir):
    output_dir = Path(output_dir)

    state_files = list(output_dir.rglob("trainer_state.json"))
    if not state_files:
        raise FileNotFoundError(f"No trainer_state.json found under: {output_dir}")

    best_state_file = None
    best_state = None
    max_global_step = -1

    for sf in state_files:
        try:
            state = json.loads(sf.read_text())
            global_step = int(state.get("global_step", -1))

            if global_step > max_global_step:
                max_global_step = global_step
                best_state_file = sf
                best_state = state
        except Exception:
            continue

    if best_state is None:
        raise RuntimeError("Could not read any valid trainer_state.json")

    # Prefer official best_model_checkpoint if available
    official_best = best_state.get("best_model_checkpoint", None)
    official_metric = best_state.get("best_metric", None)

    if official_best is not None:
        official_best_path = Path(official_best)

        if not official_best_path.exists():
            official_best_path = output_dir / official_best_path.name

        if official_best_path.exists():
            best_step = int(official_best_path.name.replace("checkpoint-", ""))
            return official_best_path, best_step, official_metric, None, best_state_file

    # Fallback: infer best checkpoint manually from eval_loss logs
    eval_rows = []

    for item in best_state.get("log_history", []):
        if "eval_loss" in item and "step" in item:
            eval_rows.append({
                "step": int(item["step"]),
                "eval_loss": float(item["eval_loss"]),
                "epoch": item.get("epoch", None),
            })

    if not eval_rows:
        raise RuntimeError("No eval_loss records found in trainer_state.json")

    best_row = min(eval_rows, key=lambda x: x["eval_loss"])

    best_step = best_row["step"]
    best_eval_loss = best_row["eval_loss"]
    best_epoch = best_row["epoch"]

    best_checkpoint_path = output_dir / f"checkpoint-{best_step}"

    if not best_checkpoint_path.exists():
        existing = sorted([p.name for p in output_dir.glob("checkpoint-*")])
        raise FileNotFoundError(
            f"Best checkpoint is missing: {best_checkpoint_path}\n"
            f"Best step from logs = {best_step}, eval_loss = {best_eval_loss}\n"
            f"Existing checkpoints: {existing}\n"
            f"Rerun with SAVE_TOTAL_LIMIT high enough."
        )

    return best_checkpoint_path, best_step, best_eval_loss, best_epoch, best_state_file


BEST_CHECKPOINT_PATH, BEST_STEP, BEST_EVAL_LOSS, BEST_EPOCH, BEST_STATE_FILE = find_best_checkpoint_from_logs(OUTPUT_DIR)

print("Best checkpoint:")
print("  path:", BEST_CHECKPOINT_PATH)
print("  step:", BEST_STEP)
print("  epoch:", BEST_EPOCH)
print("  eval_loss:", BEST_EVAL_LOSS)
print("  trainer_state:", BEST_STATE_FILE)

# ------------------------------------------------------------
# Load best checkpoint into the current trainer/model
# ------------------------------------------------------------

print("\nLoading best checkpoint into model...")

try:
    trainer._load_from_checkpoint(str(BEST_CHECKPOINT_PATH), model=trainer.model)
    model = trainer.model
    print("Loaded best checkpoint using trainer._load_from_checkpoint().")

except Exception as e:
    print("trainer._load_from_checkpoint failed:")
    print(repr(e))
    print("Trying PEFT fallback load...")

    from peft import PeftModel

    base_model, tokenizer = UnslothModel.from_pretrained(
        model_name = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype = dtype,
        load_in_4bit = LOAD_IN_4BIT,
    )

    if hasattr(tokenizer, "tokenizer"):
        tokenizer = tokenizer.tokenizer

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = PeftModel.from_pretrained(base_model, str(BEST_CHECKPOINT_PATH))
    trainer.model = model

    print("Loaded best checkpoint using PEFT fallback.")

# ------------------------------------------------------------
# Save best adapter separately
# ------------------------------------------------------------

BEST_ADAPTER_PATH = ADAPTER_DIR / f"{EXPERIMENT_NAME}_best_step{BEST_STEP}"
BEST_ADAPTER_PATH.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(BEST_ADAPTER_PATH))
tokenizer.save_pretrained(str(BEST_ADAPTER_PATH))

print("\nSaved BEST adapter to:")
print(BEST_ADAPTER_PATH)

print("\nActive model for inference is now the BEST checkpoint.")
print("BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH)
print("BEST_STEP:", BEST_STEP)
print("BEST_EVAL_LOSS:", BEST_EVAL_LOSS)

Best checkpoint:
  path: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700
  step: 700
  epoch: 1.8005148005148004
  eval_loss: 1.6058480739593506
  trainer_state: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-2000/trainer_state.json

Loading best checkpoint into model...
Loaded best checkpoint using trainer._load_from_checkpoint().


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs_best_step700/tokenizer_config.json.



Saved BEST adapter to:
/content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs_best_step700

Active model for inference is now the BEST checkpoint.
BEST_CHECKPOINT_PATH: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700
BEST_STEP: 700
BEST_EVAL_LOSS: 1.6058480739593506


### **Quick Inference**

In [20]:
# ============================================================
# Cell 18 — Quick inference function with BEAM-4 decoding
# Fixes Unsloth/Transformers beam-search cache issue using use_cache=False
# ============================================================

import torch

if "BEST_CHECKPOINT_PATH" not in globals():
    raise RuntimeError(
        "BEST_CHECKPOINT_PATH is not defined. "
        "Run Cell 17 first to load the best checkpoint before inference."
    )

print("Inference will use BEST checkpoint:")
print("BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH)
print("BEST_STEP:", BEST_STEP)
print("BEST_EVAL_LOSS:", BEST_EVAL_LOSS)

try:
    UnslothModel.for_inference(model)
except Exception as e:
    print("for_inference not available or not needed:", repr(e))

# ------------------------------------------------------------
# Beam-4 decoding config
# ------------------------------------------------------------

DECODE_TAG = "beam4"

GENERATION_KWARGS = {
    "do_sample": False,
    "num_beams": 4,
    "num_return_sequences": 1,
    "length_penalty": 1.0,
    "early_stopping": True,
    "repetition_penalty": 1.05,

    # IMPORTANT:
    # Fixes: AttributeError: 'tuple' object has no attribute 'reorder_cache'
    # Beam search needs cache reordering, but this Unsloth/Transformers setup
    # returns old-style tuple cache. Disabling cache avoids the crash.
    "use_cache": False,
}

print("Decode tag:", DECODE_TAG)
print("Generation kwargs:", GENERATION_KWARGS)

def extract_assistant_answer(decoded_text):
    if RESPONSE_MARKER in decoded_text:
        answer = decoded_text.split(RESPONSE_MARKER)[-1]
    else:
        answer = decoded_text

    special_tokens = [
        tokenizer.eos_token,
        tokenizer.pad_token,
        "<|endoftext|>",
        "<|im_end|>",
    ]

    for tok in special_tokens:
        if tok:
            answer = answer.replace(tok, "")

    # Clean known artifacts
    answer = answer.replace("<turn|>", "")
    answer = answer.strip()

    return answer

def generate_translation_from_row(row, max_new_tokens=120):
    user_text = make_user_prompt(row)

    prompt = format_sft_text(
        system_text=SYSTEM_PROMPT,
        user_text=user_text,
        assistant_text=None,
        add_eos=False,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            **GENERATION_KWARGS,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    answer = extract_assistant_answer(decoded)

    return answer, decoded

# ------------------------------------------------------------
# Quick sample check
# ------------------------------------------------------------

sample = eval_df.sample(1, random_state=SEED).iloc[0].to_dict()

pred, raw = generate_translation_from_row(sample)

print("Country/config:", sample["config"])
print("Dialect:", sample["dialect"])
print("Domain:", sample["domain"])

print("\nEnglish:")
print(sample["source_text"])

print("\nReference Arabic:")
print(sample["target_arabic"])

print("\nBeam4 Prediction:")
print(pred)

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inference will use BEST checkpoint:
BEST_CHECKPOINT_PATH: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700
BEST_STEP: 700
BEST_EVAL_LOSS: 1.6058480739593506
Decode tag: beam4
Generation kwargs: {'do_sample': False, 'num_beams': 4, 'num_return_sequences': 1, 'length_penalty': 1.0, 'early_stopping': True, 'repetition_penalty': 1.05, 'use_cache': False}


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

Country/config: EG
Dialect: Egyptian Arabic (Cairene) Dialect
Domain: Construction and real estate

English:
Engineer, good morning. Before you run your cables on the third floor, let's coordinate the wall chases.

Reference Arabic:
صباح الخير يا هندسه. قبل ما تمد الكابلات في الدور التالت، خلينا نتفق على مجاري الحيطان.

Beam4 Prediction:
يا باشمهندس صباح الخير. قبل ما تركب الكابلات في الدور التالت، خلينا نتفق على حفر الحيطان.


### Generate predictions for all eval set

In [ ]:
# ============================================================
# Cell 19 — Generate BEAM-4 predictions on FULL eval/test set
# Uses BEST checkpoint loaded in Cell 17
# Saves to a separate beam4 prediction file
# ============================================================

from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import time
import shutil
import torch

if "BEST_CHECKPOINT_PATH" not in globals():
    raise RuntimeError("Run Cell 17 first. The best checkpoint is not loaded.")

if "generate_translation_from_row" not in globals():
    raise RuntimeError("Run Cell 18 first. The beam4 generation function is not defined.")

if "DECODE_TAG" not in globals():
    DECODE_TAG = "beam4"

# None = full eval/test set
EVAL_LIMIT = None

SAVE_EVERY = 50
STORE_RAW_OUTPUT = False

# First beam4 run:
# False is fine because the beam4 file probably does not exist yet.
# If a partial beam4 file exists, this resumes it.
OVERWRITE_EXISTING = False

# Only used if OVERWRITE_EXISTING=True
BACKUP_BEFORE_OVERWRITE = True

eval_tag = f"best_step{BEST_STEP}_{DECODE_TAG}"

pred_path = PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}_{eval_tag}.csv"

print("Experiment:", EXPERIMENT_NAME)
print("Using BEST checkpoint:", BEST_CHECKPOINT_PATH)
print("Best step:", BEST_STEP)
print("Best eval_loss:", BEST_EVAL_LOSS)
print("Decode tag:", DECODE_TAG)
print("Generation kwargs:", GENERATION_KWARGS)
print("Saving predictions to:", pred_path)
print("OVERWRITE_EXISTING:", OVERWRITE_EXISTING)

full_eval_df = eval_df.reset_index(drop=True).copy()

if EVAL_LIMIT is not None:
    full_eval_df = full_eval_df.iloc[:EVAL_LIMIT].copy()

expected_n = len(full_eval_df)

print("Total eval/test examples to evaluate:", expected_n)

# ------------------------------------------------------------
# Start fresh or resume
# ------------------------------------------------------------

if pred_path.exists() and OVERWRITE_EXISTING:
    if BACKUP_BEFORE_OVERWRITE:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        backup_path = pred_path.with_suffix(f".backup_{timestamp}.csv")
        shutil.copy2(pred_path, backup_path)
        print("Existing beam4 prediction file backed up to:", backup_path)

    pred_rows = []
    done_ids = set()
    print("FORCING fresh beam4 prediction generation from scratch.")

elif pred_path.exists() and not OVERWRITE_EXISTING:
    existing_df = pd.read_csv(pred_path)

    if "source_id" in existing_df.columns:
        existing_df["source_id"] = existing_df["source_id"].astype(str)

        expected_ids = set(full_eval_df["source_id"].astype(str).tolist())
        existing_df = existing_df[existing_df["source_id"].isin(expected_ids)].copy()
        existing_df = existing_df.drop_duplicates(subset=["source_id"], keep="first")

        pred_rows = existing_df.to_dict("records")
        done_ids = set(existing_df["source_id"].astype(str).tolist())

        print(f"Resuming from existing beam4 file: {len(done_ids)} examples already done.")

    else:
        print("Existing file has no source_id column. Starting from scratch.")
        pred_rows = []
        done_ids = set()

else:
    pred_rows = []
    done_ids = set()
    print("No existing beam4 prediction file. Starting from scratch.")

# ------------------------------------------------------------
# Generate predictions
# ------------------------------------------------------------

start_time = time.time()

for _, row in tqdm(full_eval_df.iterrows(), total=len(full_eval_df)):
    row_dict = row.to_dict()
    source_id = str(row_dict["source_id"])

    if source_id in done_ids:
        continue

    try:
        pred, raw = generate_translation_from_row(row_dict)

        out_row = {
            "source_id": row_dict["source_id"],
            "config": row_dict.get("config", ""),
            "dialect": row_dict.get("dialect", ""),
            "domain": row_dict.get("domain", ""),
            "source_text": row_dict["source_text"],
            "reference_arabic": row_dict["target_arabic"],
            "prediction": pred,
            "decode_tag": DECODE_TAG,
            "generation_kwargs": str(GENERATION_KWARGS),
            "model_checkpoint": str(BEST_CHECKPOINT_PATH),
            "best_step": BEST_STEP,
            "best_eval_loss": BEST_EVAL_LOSS,
            "experiment_name": EXPERIMENT_NAME,
            "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }

        if STORE_RAW_OUTPUT:
            out_row["raw_output"] = raw

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()

        out_row = {
            "source_id": row_dict.get("source_id", ""),
            "config": row_dict.get("config", ""),
            "dialect": row_dict.get("dialect", ""),
            "domain": row_dict.get("domain", ""),
            "source_text": row_dict.get("source_text", ""),
            "reference_arabic": row_dict.get("target_arabic", ""),
            "prediction": "",
            "generation_error": repr(e),
            "decode_tag": DECODE_TAG,
            "generation_kwargs": str(GENERATION_KWARGS),
            "model_checkpoint": str(BEST_CHECKPOINT_PATH),
            "best_step": BEST_STEP,
            "best_eval_loss": BEST_EVAL_LOSS,
            "experiment_name": EXPERIMENT_NAME,
            "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }

    except Exception as e:
        out_row = {
            "source_id": row_dict.get("source_id", ""),
            "config": row_dict.get("config", ""),
            "dialect": row_dict.get("dialect", ""),
            "domain": row_dict.get("domain", ""),
            "source_text": row_dict.get("source_text", ""),
            "reference_arabic": row_dict.get("target_arabic", ""),
            "prediction": "",
            "generation_error": repr(e),
            "decode_tag": DECODE_TAG,
            "generation_kwargs": str(GENERATION_KWARGS),
            "model_checkpoint": str(BEST_CHECKPOINT_PATH),
            "best_step": BEST_STEP,
            "best_eval_loss": BEST_EVAL_LOSS,
            "experiment_name": EXPERIMENT_NAME,
            "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }

    pred_rows.append(out_row)
    done_ids.add(source_id)

    if len(pred_rows) % SAVE_EVERY == 0:
        tmp_df = pd.DataFrame(pred_rows)
        tmp_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
        print(f"Saved partial beam4 predictions: {len(tmp_df)} rows")

# ------------------------------------------------------------
# Final save
# ------------------------------------------------------------

pred_df = pd.DataFrame(pred_rows)

order_df = full_eval_df[["source_id"]].copy()
order_df["source_id"] = order_df["source_id"].astype(str)
order_df["eval_order"] = range(len(order_df))

pred_df["source_id"] = pred_df["source_id"].astype(str)
pred_df = pred_df.merge(order_df, on="source_id", how="left")
pred_df = pred_df.sort_values("eval_order").drop(columns=["eval_order"])
pred_df = pred_df.reset_index(drop=True)

pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

elapsed = time.time() - start_time

expected_ids = set(full_eval_df["source_id"].astype(str).tolist())
actual_ids = set(pred_df["source_id"].astype(str).tolist())

missing_ids = expected_ids - actual_ids
extra_ids = actual_ids - expected_ids

print("\nDone.")
print("Saved beam4 predictions to:", pred_path)
print("Total rows saved:", len(pred_df))
print("Expected eval/test rows:", expected_n)
print(f"Elapsed time: {elapsed / 60:.2f} minutes")

if missing_ids:
    raise RuntimeError(f"Prediction file is incomplete. Missing {len(missing_ids)} eval examples.")

if extra_ids:
    print(f"Warning: prediction file has {len(extra_ids)} extra source_ids not in current eval_df.")

if len(pred_df) == expected_n:
    print("Full eval/test set was evaluated successfully with beam4.")
else:
    print("Warning: row count differs from expected eval size.")

display(pred_df.head())

Experiment: qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs
Using BEST checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700
Best step: 700
Best eval_loss: 1.6058480739593506
Decode tag: beam4
Generation kwargs: {'do_sample': False, 'num_beams': 4, 'num_return_sequences': 1, 'length_penalty': 1.0, 'early_stopping': True, 'repetition_penalty': 1.05, 'use_cache': False}
Saving predictions to: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs_best_step700_beam4.csv
OVERWRITE_EXISTING: False
Total eval/test examples to evaluate: 1118
No existing beam4 prediction file. Starting from scratch.


  0%|          | 0/1118 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Saved partial beam4 predictions: 50 rows


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Saved partial beam4 predictions: 300 rows


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Saved partial beam4 predictions: 350 rows


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

### **Compute BLEU and chrF**

In [ ]:
# ============================================================
# Cell 20 — Compute BLEU, spBLEU, chrF, chrF++ for BEAM-4 predictions
# Uses predictions from BEST checkpoint + beam4 decoding
# ============================================================

import pandas as pd
import json
from pathlib import Path

try:
    import sacrebleu
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"])
    import sacrebleu

if "BEST_STEP" not in globals():
    raise RuntimeError("BEST_STEP is not defined. Run Cell 17 first.")

if "DECODE_TAG" not in globals():
    DECODE_TAG = "beam4"

eval_tag = f"best_step{BEST_STEP}_{DECODE_TAG}"

pred_path = PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}_{eval_tag}.csv"
metrics_path = PRED_DIR / f"full_eval_metrics_{EXPERIMENT_NAME}_{eval_tag}.json"

print("Experiment:", EXPERIMENT_NAME)
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Best step:", BEST_STEP)
print("Decode tag:", DECODE_TAG)
print("Prediction file:", pred_path)
print("Metrics file:", metrics_path)

if not pred_path.exists():
    raise FileNotFoundError(f"Prediction file not found: {pred_path}")

pred_df = pd.read_csv(pred_path)

required_cols = {"source_id", "prediction", "reference_arabic"}
missing = required_cols - set(pred_df.columns)

if missing:
    raise ValueError(f"Missing required columns in prediction file: {missing}")

pred_df["source_id"] = pred_df["source_id"].astype(str)
pred_df["prediction"] = pred_df["prediction"].fillna("").astype(str)
pred_df["reference_arabic"] = pred_df["reference_arabic"].fillna("").astype(str)

expected_eval_df = eval_df.reset_index(drop=True).copy()
expected_eval_df["source_id"] = expected_eval_df["source_id"].astype(str)

expected_ids = set(expected_eval_df["source_id"].tolist())
actual_ids = set(pred_df["source_id"].tolist())

missing_ids = expected_ids - actual_ids
extra_ids = actual_ids - expected_ids

print("\n==============================")
print("Full Eval/Test Coverage Check")
print("==============================")
print("Expected eval/test examples:", len(expected_eval_df))
print("Prediction rows:", len(pred_df))
print("Unique prediction source_ids:", len(actual_ids))

if missing_ids:
    raise RuntimeError(
        f"Prediction file is NOT full eval/test. "
        f"Missing {len(missing_ids)} examples. "
        f"Run Cell 19 again to finish generation."
    )

if extra_ids:
    print(f"Warning: prediction file has {len(extra_ids)} extra source_ids not in current eval_df.")

print("Full eval/test coverage confirmed.")

order_df = expected_eval_df[["source_id"]].copy()
pred_df_ordered = order_df.merge(pred_df, on="source_id", how="left")

preds = pred_df_ordered["prediction"].fillna("").astype(str).tolist()
refs = pred_df_ordered["reference_arabic"].fillna("").astype(str).tolist()

# BLEU
bleu = sacrebleu.corpus_bleu(preds, [refs])

# spBLEU using FLORES200 tokenizer
try:
    spbleu = sacrebleu.corpus_bleu(preds, [refs], tokenize="flores200")
except Exception as e:
    print("spBLEU with flores200 failed:", repr(e))
    print("Installing sentencepiece and retrying...")
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"])
    spbleu = sacrebleu.corpus_bleu(preds, [refs], tokenize="flores200")

# chrF and chrF++
chrf = sacrebleu.corpus_chrf(preds, [refs])
chrfpp = sacrebleu.corpus_chrf(preds, [refs], word_order=2)

metrics = {
    "experiment": EXPERIMENT_NAME,
    "checkpoint": str(BEST_CHECKPOINT_PATH),
    "best_step": int(BEST_STEP),
    "best_eval_loss": float(BEST_EVAL_LOSS),
    "decode_tag": DECODE_TAG,
    "generation_kwargs": str(GENERATION_KWARGS) if "GENERATION_KWARGS" in globals() else "",
    "num_examples": len(pred_df_ordered),
    "BLEU": bleu.score,
    "spBLEU": spbleu.score,
    "chrF": chrf.score,
    "chrF++": chrfpp.score,
    "prediction_file": str(pred_path),
}

print("\n==============================")
print("Full Eval/Test Metrics from BEST checkpoint + BEAM-4")
print("==============================")
print(f"Examples: {len(pred_df_ordered)}")
print(f"BLEU:     {bleu.score:.4f}")
print(f"spBLEU:   {spbleu.score:.4f}")
print(f"chrF:     {chrf.score:.4f}")
print(f"chrF++:   {chrfpp.score:.4f}")

def compute_group_metrics(df, group_col):
    rows = []

    if group_col not in df.columns:
        return pd.DataFrame(rows)

    for group_value in sorted(df[group_col].dropna().unique()):
        tmp = df[df[group_col] == group_value]

        if len(tmp) == 0:
            continue

        group_preds = tmp["prediction"].fillna("").astype(str).tolist()
        group_refs = tmp["reference_arabic"].fillna("").astype(str).tolist()

        try:
            group_spbleu = sacrebleu.corpus_bleu(
                group_preds,
                [group_refs],
                tokenize="flores200",
            ).score
        except Exception:
            group_spbleu = None

        rows.append({
            group_col: group_value,
            "num_examples": len(tmp),
            "BLEU": sacrebleu.corpus_bleu(group_preds, [group_refs]).score,
            "spBLEU": group_spbleu,
            "chrF": sacrebleu.corpus_chrf(group_preds, [group_refs]).score,
            "chrF++": sacrebleu.corpus_chrf(
                group_preds,
                [group_refs],
                word_order=2,
            ).score,
        })

    return pd.DataFrame(rows)

per_config_df = compute_group_metrics(pred_df_ordered, "config")
per_dialect_df = compute_group_metrics(pred_df_ordered, "dialect")
per_domain_df = compute_group_metrics(pred_df_ordered, "domain")

if len(per_config_df):
    print("\n==============================")
    print("Per-config Metrics")
    print("==============================")
    display(per_config_df)
    metrics["per_config"] = per_config_df.to_dict("records")

if len(per_dialect_df):
    print("\n==============================")
    print("Per-dialect Metrics")
    print("==============================")
    display(per_dialect_df)
    metrics["per_dialect"] = per_dialect_df.to_dict("records")

if len(per_domain_df):
    print("\n==============================")
    print("Per-domain Metrics")
    print("==============================")
    display(per_domain_df)
    metrics["per_domain"] = per_domain_df.to_dict("records")

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("\nSaved beam4 metrics to:")
print(metrics_path)

display(pred_df_ordered[["source_text", "reference_arabic", "prediction"]].head(10))